# 04 — WOE / IV Analysis

With only 11 raw features (by design — see PROGRESS.md for the dataset choice),
this notebook isn't about aggressive feature *elimination* — there's little to
cut. It's about producing the classic credit-scoring artifact (a full,
inspectable IV table) and testing whether a WOE-encoded logistic regression —
the traditional scorecard approach — is competitive with the raw-feature models
from notebooks 02-03.

In [1]:
import sys, json, os
sys.path.append('..')

import pandas as pd
from optbinning import BinningProcess
from sklearn.linear_model import LogisticRegression

from src.preprocessing import TARGET, CATEGORICAL_COLS, NUMERIC_COLS
from src.metrics import summarize

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

In [2]:
train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val, y_val = val.drop(columns=[TARGET]), val[TARGET]
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]

variable_names = list(X_train.columns)
categorical_variables = CATEGORICAL_COLS

binning = BinningProcess(variable_names, categorical_variables=categorical_variables)
binning.fit(X_train, y_train)

/Users/sonaliverma/Downloads/Data Science/Projects/Credit Risk/.venv/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/sonaliverma/Downloads/Data Science/Projects/Credit Risk/.venv/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/sonaliverma/Downloads/Data Science/Projects/Credit Risk/.venv/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/sonaliverma/Downloads/Data Science/Projects/Credit Risk/.venv/lib/python3.13/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  war

,variable_names,"['person_age', 'person_income', ...]"
,max_n_prebins,20
,min_prebin_size,0.05
,min_n_bins,None
,max_n_bins,None
,min_bin_size,None
,max_bin_size,None
,max_pvalue,None
,max_pvalue_policy,'consecutive'
,selection_criteria,None
,fixed_variables,None


## IV table — full, inspectable at a glance

In [3]:
summary = binning.summary().sort_values('iv', ascending=False)
print(summary[['name', 'dtype', 'n_bins', 'iv']].to_string(index=False))

os.makedirs('../reports', exist_ok=True)
summary.to_csv('../reports/iv_table.csv', index=False)

                      name       dtype n_bins        iv
       loan_percent_income   numerical      9  0.933647
                loan_grade categorical      4  0.864474
             loan_int_rate   numerical     12  0.775623
             person_income   numerical      9  0.525737
     person_home_ownership categorical      3  0.359705
 cb_person_default_on_file categorical      2   0.17045
                 loan_amnt   numerical     10  0.100319
               loan_intent categorical      6  0.098351
         person_emp_length   numerical      7  0.060319
                person_age   numerical      8  0.013242
cb_person_cred_hist_length   numerical      5  0.004913


**Interpretation guide (from the project plan):** <0.02 useless, 0.02-0.1 weak,
0.1-0.3 medium, 0.3-0.5 strong, >0.5 suspiciously strong (check for leakage).

Expect `loan_grade` and/or `loan_percent_income`/`loan_int_rate` to land in the
>0.5 band — per EDA and the ablation model (notebook 03), this is legitimate
(underwriting-time information, not a post-outcome leak), not a bug.

## WOE-transform all splits (fit on train only)

In [4]:
X_train_woe = binning.transform(X_train, metric='woe')
X_val_woe = binning.transform(X_val, metric='woe')
X_test_woe = binning.transform(X_test, metric='woe')
print('WOE-transformed shape (1 column per variable, no one-hot expansion):', X_train_woe.shape)
X_train_woe.head()

WOE-transformed shape (1 column per variable, no one-hot expansion): (19449, 11)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,0.027815,-0.175268,-0.493357,-0.354840,0.093882,0.384199,-0.416125,0.306411,-2.230833,0.218613,0.080144
1,-0.251175,-0.175268,-0.493357,0.159893,0.290010,0.384199,-0.265692,0.371347,-2.067846,0.218613,-0.018824
2,0.052805,0.949525,0.634146,0.429521,0.498391,0.921057,-0.416125,0.904123,0.206011,0.218613,0.059188
3,-0.251175,-0.175268,0.634146,0.133970,0.290010,0.051378,0.001038,0.019275,0.881999,-0.790831,-0.018824
4,0.027815,-0.767875,-0.493357,-0.354840,-0.281398,-1.748541,0.219770,-1.817678,0.155802,0.218613,0.080144


## WOE-encoded logistic regression — the classic scorecard model

Compare directly against the raw-feature LR baseline from notebook 02.

In [5]:
lr_woe = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
lr_woe.fit(X_train_woe, y_train)

proba_woe_val = lr_woe.predict_proba(X_val_woe)[:, 1]
proba_woe_test = lr_woe.predict_proba(X_test_woe)[:, 1]

baseline = json.load(open('../reports/baseline_results.json'))['test']
comparison = pd.DataFrame({
    'lr_woe_encoded': summarize(y_test, proba_woe_test),
    'lr_raw_onehot_baseline': baseline,
}).T
print(comparison.round(4))

                           auc      ks    gini  pr_auc   brier
lr_woe_encoded          0.8853  0.6534  0.7707  0.7537  0.1282
lr_raw_onehot_baseline  0.8697  0.6038  0.7394  0.7166  0.1384


In [6]:
results = {
    'iv_table': summary[['name', 'iv']].to_dict(orient='records'),
    'lr_woe_encoded': {'val': summarize(y_val, proba_woe_val), 'test': summarize(y_test, proba_woe_test)},
}
with open('../reports/woe_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

for name in X_train_woe.columns:
    X_train_woe.rename(columns={name: name}, inplace=True)  # no-op, keeps columns explicit
X_train_woe[TARGET] = y_train.values
X_val_woe[TARGET] = y_val.values
X_test_woe[TARGET] = y_test.values
X_train_woe.to_csv('../data/processed/train_woe.csv', index=False)
X_val_woe.to_csv('../data/processed/val_woe.csv', index=False)
X_test_woe.to_csv('../data/processed/test_woe.csv', index=False)
print('saved reports/woe_results.json and data/processed/{train,val,test}_woe.csv')

saved reports/woe_results.json and data/processed/{train,val,test}_woe.csv


## Summary

- Full IV table computed and saved to `reports/iv_table.csv` — every one of the 11
  features listed with its IV, small enough to include in full in the README.
- WOE-encoded logistic regression compared directly against the raw-feature baseline
  — confirms whether WOE encoding helps a linear model here (it's most valuable when
  raw features are non-monotonic or have many sparse categorical levels — with this
  dataset's low cardinality, the gain may be small, and that's a legitimate, reportable
  finding either way, not a failure).
- WOE-transformed splits saved for optional reuse.